# 15 — Linear Attention

## Goal

Standard self-attention explicitly compares every query position with
every key position.

For a sequence of length $T$, the attention-score matrix has shape

$$
(T,T).
$$

This creates quadratic cost with respect to sequence length.

This lesson studies a different formulation of attention based on
kernel feature maps.

The main questions are:

1. why standard attention constructs a $T \times T$ matrix,
2. how attention can be rewritten using associativity,
3. what role a feature map $\phi(\cdot)$ plays,
4. how causal linear attention requires prefix states,
5. why linear attention changes both computation and model behavior,
6. and how its recurrent state differs from a Transformer KV cache.

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F


## 1. Why Standard Attention Is Quadratic

For one attention head,

$$
Q,K,V
\in
\mathbb{R}^{T\times D}.
$$

The query-key similarity matrix is

$$
QK^\top
\in
\mathbb{R}^{T\times T}.
$$

Every query position is compared with every key position.

The number of query-key comparisons therefore grows as

$$
T^2.
$$

The attention matrix itself also requires memory proportional to

$$
T^2.
$$

This quadratic interaction is the main sequence-length bottleneck of
standard attention.

In [2]:
sequence_length: int = 4
head_dim: int = 3

q: torch.Tensor = torch.randn(
    sequence_length,
    head_dim,
)

k: torch.Tensor = torch.randn(
    sequence_length,
    head_dim,
)

v: torch.Tensor = torch.randn(
    sequence_length,
    head_dim,
)

scores: torch.Tensor = q @ k.transpose(-2, -1)

print("Q:", q.shape)
print("K:", k.shape)
print("scores:", scores.shape)

Q: torch.Size([4, 3])
K: torch.Size([4, 3])
scores: torch.Size([4, 4])


## 2. Reordering the Matrix Multiplications

Ignoring softmax temporarily, consider

$$
Y=(QK^\top)V.
$$

Matrix multiplication is associative:

$$
(AB)C=A(BC).
$$

Therefore,

$$
(QK^\top)V
=
Q(K^\top V).
$$

The standard order creates

$$
QK^\top
\in
\mathbb{R}^{T\times T}.
$$

The reordered computation first creates

$$
K^\top V
\in
\mathbb{R}^{D\times D},
$$

and then computes

$$
Q(K^\top V)
\in
\mathbb{R}^{T\times D}.
$$

The $T\times T$ matrix never needs to be materialized.

In [3]:
standard_order: torch.Tensor = (q @ k.transpose(-2, -1)) @ v

reordered: torch.Tensor = q @ (k.transpose(-2, -1) @ v)

print(
    torch.allclose(
        standard_order,
        reordered,
        atol=1e-6,
    )
)

True


## 3. From Pairwise Attention to a Feature Map

Standard softmax attention for one query position can be written as

$$
y_i
=
\frac{
\sum_j
\exp(q_i^\top k_j)
v_j
}{
\sum_j
\exp(q_i^\top k_j)
}.
$$

The difficulty is that the similarity

$$
\exp(q_i^\top k_j)
$$

is computed separately for every query-key pair.

Suppose instead that the similarity function can be written as an inner
product between transformed features:

$$
\operatorname{sim}(q,k)
=
\phi(q)^\top\phi(k).
$$

Then attention becomes

$$
y_i
=
\frac{
\sum_j
\phi(q_i)^\top\phi(k_j)v_j
}{
\sum_j
\phi(q_i)^\top\phi(k_j)
}.
$$

Because $\phi(q_i)$ does not depend on $j$, it can be moved outside the
sum:

$$
y_i
=
\frac{
\phi(q_i)^\top
\left(
\sum_j
\phi(k_j)v_j^\top
\right)
}{
\phi(q_i)^\top
\left(
\sum_j
\phi(k_j)
\right)
}.
$$

This is the key algebraic step behind kernelized linear attention.

> **Important:** this algebra does not make kernelized linear attention
> identical to ordinary softmax attention.

The feature map $\phi$ defines a different similarity function unless it
is specifically chosen to reproduce or approximate the softmax kernel.

The linear-time computation comes from the factorization

$$
\operatorname{sim}(q,k)
=
\phi(q)^\top\phi(k),
$$

not from simply rearranging ordinary softmax attention.

In [4]:
def positive_feature_map(x: torch.Tensor) -> torch.Tensor:
    """Apply the positive feature map ELU(x) + 1.

    Args:
        x: Input tensor of any shape.

    Returns:
        Positive transformed tensor with the same shape as `x`.
    """
    return torch.where(x > 0, x + 1.0, torch.exp(x))

In [5]:
x_test: torch.Tensor = torch.tensor([-2.0, -1.0, 0.0, 1.0, 2.0])

phi_test: torch.Tensor = positive_feature_map(x_test)

print(phi_test)

tensor([0.1353, 0.3679, 1.0000, 2.0000, 3.0000])


In [6]:
def kernel_attention_pairwise(
    q: torch.Tensor, k: torch.Tensor, v: torch.Tensor, epsilon: float = 1e-6
) -> torch.Tensor:
    """Compute normalized kernel attention using explicit pairwise scores.

    Args:
        q: Query tensor with shape `(T, D)`.
        k: Key tensor with shape `(T, D)`.
        v: Value tensor with shape `(T, D_V)`.
        epsilon: Small numerical-stability constant.

    Returns:
        Attention output with shape `(T, D_V)`.
    """
    q_features: torch.Tensor = positive_feature_map(q)
    k_features: torch.Tensor = positive_feature_map(k)

    # Explicit pairwise similarities: (T,D)@(D,T) -> (T,T)

    similarities: torch.Tensor = q_features @ k_features.transpose(-2, -1)

    # Sum similarities across all keys for each query.
    normalizer: torch.Tensor = similarities.sum(dim=-1, keepdim=True)

    # Weighted sum of values followed by normalization.

    return (similarities @ v) / (normalizer + epsilon)


In [7]:
def kernel_attention_linear(
    q: torch.Tensor, k: torch.Tensor, v: torch.Tensor, epsilon: float = 1e-6
) -> torch.Tensor:
    """Compute normalized kernel attention without a T x T matrix.

    Args:
        q: Query tensor with shape `(T, D)`.
        k: Key tensor with shape `(T, D)`.
        v: Value tensor with shape `(T, D_V)`.
        epsilon: Small numerical-stability constant.

    Returns:
        Attention output with shape `(T, D_V)`.
    """

    q_features: torch.Tensor = positive_feature_map(q)
    k_features: torch.Tensor = positive_feature_map(k)

    # Summarize all key-value interactions into a fixed-size matrix.
    #
    # (D, T) @ (T, D_V)
    #       ->
    # (D, D_V)
    kv_summary: torch.Tensor = k_features.transpose(-2, -1) @ v

    # Summarize all keys for the normalization term.
    # (T, D) -> (D)
    key_summary: torch.Tensor = k_features.sum(dim=0)

    # Numerator: (T, D) @ (D, D_V) -> (T, D_V)
    numerator: torch.Tensor = q_features @ kv_summary

    # Denominator: (T, D) @ (D, 1) -> (T, 1)
    denominator: torch.Tensor = q_features @ key_summary.unsqueeze(-1)

    return numerator / (denominator + epsilon)

In [8]:
sequence_length: int = 4
head_dim: int = 3

q: torch.Tensor = torch.randn(
    sequence_length,
    head_dim,
)

k: torch.Tensor = torch.randn(
    sequence_length,
    head_dim,
)

v: torch.Tensor = torch.randn(
    sequence_length,
    head_dim,
)

In [9]:
pairwise_output: torch.Tensor = kernel_attention_pairwise(
    q,
    k,
    v,
)

linear_output: torch.Tensor = kernel_attention_linear(
    q,
    k,
    v,
)

print(
    "pairwise shape:",
    pairwise_output.shape,
)

print(
    "linear shape:",
    linear_output.shape,
)

print(
    "outputs match:",
    torch.allclose(
        pairwise_output,
        linear_output,
        atol=1e-5,
    ),
)

pairwise shape: torch.Size([4, 3])
linear shape: torch.Size([4, 3])
outputs match: True


### What Became Linear?

The sequence-length dependence changed.

Pairwise kernel attention explicitly constructs

$$
\Phi(Q)\Phi(K)^\top
\in
\mathbb{R}^{T\times T},
$$

so its cost grows quadratically with $T$.

The reordered form instead constructs the fixed-size summary

$$
S
=
\Phi(K)^\top V.
$$

If the feature dimension and value dimension remain fixed, the cost grows
linearly with sequence length:

$$
O(T)
$$

with respect to $T$.

The important idea is not that all computation became constant-time.

The important idea is that we removed the explicit all-pairs interaction
over sequence positions.

## 4. Causal Linear Attention as a Recurrent State

The previous linear-attention formulation summarized all key/value pairs:

$$
S
=
\sum_j
\phi(k_j)v_j^\top,
$$

and

$$
z
=
\sum_j
\phi(k_j).
$$

That formulation is not causal because every query can access information
from every position.

For autoregressive language modeling, position $t$ may only use positions

$$
j \le t.
$$

We therefore define prefix states:

$$
S_t
=
\sum_{j=1}^{t}
\phi(k_j)v_j^\top,
$$

and

$$
z_t
=
\sum_{j=1}^{t}
\phi(k_j).
$$

The output at position $t$ is

$$
y_t
=
\frac{
\phi(q_t)^\top S_t
}{
\phi(q_t)^\top z_t
}.
$$

Because the prefix states can be updated recursively,

$$
S_t
=
S_{t-1}
+
\phi(k_t)v_t^\top,
$$

and

$$
z_t
=
z_{t-1}
+
\phi(k_t),
$$

causal linear attention can be viewed as a recurrent state-update system.

In [10]:
def causal_linear_attention_recurrent(
    q: torch.Tensor, k: torch.Tensor, v: torch.Tensor, epsilon: float = 1e-6
) -> torch.Tensor:
    """Compute causal linear attention with recurrent state updates.

    Args:
        q: Query tensor with shape `(T, D)`.
        k: Key tensor with shape `(T, D)`.
        v: Value tensor with shape `(T, D_V)`.
        epsilon: Small numerical-stability constant.

    Returns:
        Causal attention output with shape `(T, D_V)`.
    """

    if q.ndim != 2 or k.ndim != 2 or v.ndim != 2:
        raise ValueError("qkv must allbe rank-2 tensors")

    if q.shape != k.shape:
        raise ValueError("q & k must have the identical shapes.")

    if q.shape[0] != v.shape[0]:
        raise ValueError("qkv must have the same seq length.")

    sequence_length: int = q.shape[0]
    feature_dim: int = q.shape[1]
    value_dim: int = v.shape[1]

    q_features: torch.Tensor = positive_feature_map(q)
    k_features: torch.Tensor = positive_feature_map(k)

    # S_t stores accumulated key-value outer products.
    kv_state: torch.Tensor = torch.zeros(
        feature_dim,
        value_dim,
        dtype=q.dtype,
        device=q.device,
    )

    # z_t stores accumulated transformed keys.
    key_state: torch.Tensor = torch.zeros(
        feature_dim,
        dtype=q.dtype,
        device=q.device,
    )

    outputs: list[torch.Tensor] = []

    for position in range(sequence_length):
        q_t: torch.Tensor = q_features[position]
        k_t: torch.Tensor = k_features[position]
        v_t: torch.Tensor = v[position]

        # Outer product: (D, 1) @ (1, D_V) -> (D, D_V)
        kv_update: torch.Tensor = k_t.unsqueeze(-1) @ v_t.unsqueeze(0)

        # Include the current token before querying the state so that position t may attend to itself.
        kv_state = kv_state + kv_update
        key_state = key_state + k_t

        # (D) @ (D, D_V) -> (D_V)
        numerator: torch.Tensor = q_t @ kv_state

        # (D) @ (D) -> scalar
        denominator: torch.Tensor = q_t @ key_state

        y_t: torch.Tensor = numerator / (denominator + epsilon)

        outputs.append(y_t)

    return torch.stack(outputs, dim=0)

In [11]:
sequence_length: int = 4
feature_dim: int = 3
value_dim: int = 5

q: torch.Tensor = torch.randn(
    sequence_length,
    feature_dim,
)

k: torch.Tensor = torch.randn(
    sequence_length,
    feature_dim,
)

v: torch.Tensor = torch.randn(
    sequence_length,
    value_dim,
)

recurrent_output: torch.Tensor = causal_linear_attention_recurrent(
    q,
    k,
    v,
)

print("Q:", q.shape)
print("K:", k.shape)
print("V:", v.shape)
print("output:", recurrent_output.shape)

Q: torch.Size([4, 3])
K: torch.Size([4, 3])
V: torch.Size([4, 5])
output: torch.Size([4, 5])


In [12]:
def causal_kernel_attention_pairwise(
    q: torch.Tensor,
    k: torch.Tensor,
    v: torch.Tensor,
    epsilon: float = 1e-6,
) -> torch.Tensor:
    """Compute causal kernel attention with an explicit T x T matrix.

    This implementation is intentionally quadratic and serves as a
    correctness reference for recurrent causal linear attention.

    Args:
        q: Query tensor with shape `(T, D)`.
        k: Key tensor with shape `(T, D)`.
        v: Value tensor with shape `(T, D_V)`.
        epsilon: Small numerical-stability constant.

    Returns:
        Causal attention output with shape `(T, D_V)`.
    """
    q_features: torch.Tensor = positive_feature_map(q)
    k_features: torch.Tensor = positive_feature_map(k)

    # Explicit pairwise kernel similarities.
    #
    # (T, D) @ (D, T)
    #       ->
    # (T, T)
    similarities: torch.Tensor = q_features @ k_features.transpose(-2, -1)

    sequence_length: int = q.shape[0]

    positions: torch.Tensor = torch.arange(
        sequence_length,
        device=q.device,
    )

    query_positions: torch.Tensor = positions.unsqueeze(1)

    key_positions: torch.Tensor = positions.unsqueeze(0)

    # Entry (i, j) is valid only when key position j is not in the
    # future relative to query position i.
    causal_mask: torch.Tensor = key_positions <= query_positions

    similarities = similarities.masked_fill(
        ~causal_mask,
        0.0,
    )

    normalizer: torch.Tensor = similarities.sum(
        dim=-1,
        keepdim=True,
    )

    return (similarities @ v) / (normalizer + epsilon)

In [13]:
pairwise_causal_output: torch.Tensor = causal_kernel_attention_pairwise(
    q,
    k,
    v,
)

recurrent_output: torch.Tensor = causal_linear_attention_recurrent(
    q,
    k,
    v,
)

print(
    "outputs match:",
    torch.allclose(
        pairwise_causal_output,
        recurrent_output,
        atol=1e-5,
    ),
)

print(
    "maximum difference:",
    (pairwise_causal_output - recurrent_output).abs().max().item(),
)

outputs match: True
maximum difference: 1.1920928955078125e-07


### Recurrent Interpretation

Causal linear attention can be written as a recurrent system:

$$
S_t
=
S_{t-1}
+
\phi(k_t)v_t^\top,
$$

$$
z_t
=
z_{t-1}
+
\phi(k_t),
$$

and

$$
y_t
=
\frac{
\phi(q_t)^\top S_t
}{
\phi(q_t)^\top z_t
}.
$$

The state

$$
(S_t,z_t)
$$

has a size independent of sequence length.

This differs fundamentally from a standard Transformer KV cache, which
stores each historical key and value separately.

The benefit is a fixed-size recurrent state.

The cost is that historical information is compressed into a shared
state rather than remaining individually addressable.

## 5. Parallel Causal Linear Attention

The recurrent formulation updates the linear-attention state one token at
a time:

$$
S_t
=
S_{t-1}
+
\phi(k_t)v_t^\top,
$$

and

$$
z_t
=
z_{t-1}
+
\phi(k_t).
$$

This is natural for autoregressive inference, but training usually
processes an entire sequence at once.

The same states can also be written as prefix sums:

$$
S_t
=
\sum_{j=1}^{t}
\phi(k_j)v_j^\top,
$$

and

$$
z_t
=
\sum_{j=1}^{t}
\phi(k_j).
$$

Therefore, all prefix states can be constructed for the full sequence
during training.

This gives two execution views of the same causal linear-attention
mechanism:

<pre>
training
full sequence
    ↓
parallel prefix states

inference
one new token
    ↓
recurrent state update
</pre>

In [14]:
q_features: torch.Tensor = positive_feature_map(q)
k_features: torch.Tensor = positive_feature_map(k)

kv_updates: torch.Tensor = k_features.unsqueeze(-1) @ v.unsqueeze(-2)

print("K features:", k_features.shape)
print("V:", v.shape)
print("KV updates:", kv_updates.shape)

K features: torch.Size([4, 3])
V: torch.Size([4, 5])
KV updates: torch.Size([4, 3, 5])


In [15]:
kv_prefix_states: torch.Tensor = torch.cumsum(
    kv_updates,
    dim=0,
)

print(kv_prefix_states.shape)

torch.Size([4, 3, 5])


In [16]:
key_prefix_states: torch.Tensor = torch.cumsum(
    k_features,
    dim=0,
)

print(key_prefix_states.shape)

torch.Size([4, 3])


In [17]:
def causal_linear_attention_parallel(
    q: torch.Tensor,
    k: torch.Tensor,
    v: torch.Tensor,
    epsilon: float = 1e-6,
) -> torch.Tensor:
    """Compute causal linear attention using parallel prefix states.

    Args:
        q: Query tensor with shape `(T, D)`.
        k: Key tensor with shape `(T, D)`.
        v: Value tensor with shape `(T, D_V)`.
        epsilon: Small numerical-stability constant.

    Returns:
        Causal attention output with shape `(T, D_V)`.
    """
    if q.ndim != 2 or k.ndim != 2 or v.ndim != 2:
        raise ValueError("q, k, and v must all be rank-2 tensors.")

    if q.shape != k.shape:
        raise ValueError("q and k must have identical shapes.")

    if q.shape[0] != v.shape[0]:
        raise ValueError("q, k, and v must have the same sequence length.")

    q_features: torch.Tensor = positive_feature_map(q)
    k_features: torch.Tensor = positive_feature_map(k)

    # Construct one key-value outer-product update per position.
    #
    # (T, D, 1) @ (T, 1, D_V)
    #           ->
    # (T, D, D_V)
    kv_updates: torch.Tensor = k_features.unsqueeze(-1) @ v.unsqueeze(-2)

    # Prefix memory states:
    #
    # S_t = sum_{j <= t} phi(k_j) v_j^T
    kv_prefix_states: torch.Tensor = torch.cumsum(
        kv_updates,
        dim=0,
    )

    # Prefix normalization states:
    #
    # z_t = sum_{j <= t} phi(k_j)
    key_prefix_states: torch.Tensor = torch.cumsum(
        k_features,
        dim=0,
    )

    # Each query reads from the prefix memory available at its position.
    #
    # (T, 1, D) @ (T, D, D_V)
    #           ->
    # (T, 1, D_V)
    #           ->
    # (T, D_V)
    numerator: torch.Tensor = (
        q_features.unsqueeze(1) @ kv_prefix_states
    ).squeeze(1)

    # (T, 1, D) @ (T, D, 1)
    #           ->
    # (T, 1, 1)
    #           ->
    # (T, 1)
    denominator: torch.Tensor = (
        q_features.unsqueeze(1) @ key_prefix_states.unsqueeze(-1)
    ).squeeze(-1)

    return numerator / (denominator + epsilon)

In [18]:
pairwise_output: torch.Tensor = causal_kernel_attention_pairwise(
    q,
    k,
    v,
)

recurrent_output: torch.Tensor = causal_linear_attention_recurrent(
    q,
    k,
    v,
)

parallel_output: torch.Tensor = causal_linear_attention_parallel(
    q,
    k,
    v,
)

print(
    "pairwise vs recurrent:",
    torch.allclose(
        pairwise_output,
        recurrent_output,
        atol=1e-5,
    ),
)

print(
    "pairwise vs parallel:",
    torch.allclose(
        pairwise_output,
        parallel_output,
        atol=1e-5,
    ),
)

print(
    "recurrent vs parallel:",
    torch.allclose(
        recurrent_output,
        parallel_output,
        atol=1e-5,
    ),
)

pairwise vs recurrent: True
pairwise vs parallel: True
recurrent vs parallel: True


### Training Memory and Inference State Are Different Questions

The recurrent inference form only needs the current state:

$$
S_t \in \mathbb{R}^{D\times D_V},
$$

and

$$
z_t \in \mathbb{R}^{D}.
$$

Its persistent state therefore does not grow with sequence length.

The simple parallel training implementation above materializes all prefix
states:

$$
(S_1,S_2,\ldots,S_T),
$$

which requires memory proportional to

$$
O(TDD_V).
$$

Therefore:

<pre>
recurrent inference
→ fixed-size persistent state

parallel training
→ multiple prefix states may be materialized
</pre>

Linear sequence complexity does not imply constant training memory.

## 6. Memory Interference in a Fixed-Size State

Causal linear attention compresses historical key/value pairs into a
fixed-size state:

$$
S
=
\sum_j
\phi(k_j)v_j^\top,
$$

together with the normalization state

$$
z
=
\sum_j
\phi(k_j).
$$

A query reads from this state through

$$
y(q)
=
\frac{
\phi(q)^\top S
}{
\phi(q)^\top z
}.
$$

Expanding the numerator gives

$$
\phi(q)^\top S
=
\sum_j
\left[
\phi(q)^\top\phi(k_j)
\right]
v_j.
$$

Therefore, a query does not retrieve a value from a separate memory
slot.

It retrieves a weighted mixture of every value whose key overlaps with
the query in feature space.

This creates the possibility of memory interference.

In [19]:
def build_linear_memory(
    feature_keys: torch.Tensor,
    values: torch.Tensor,
) -> tuple[torch.Tensor, torch.Tensor]:
    """Build the fixed-size state used by normalized linear attention.

    Args:
        feature_keys: Key features with shape `(N, M)`.
        values: Values with shape `(N, D_V)`.

    Returns:
        A tuple containing:
        - KV state with shape `(M, D_V)`,
        - key normalization state with shape `(M)`.
    """
    if feature_keys.ndim != 2 or values.ndim != 2:
        raise ValueError("feature_keys and values must be rank-2 tensors.")

    if feature_keys.shape[0] != values.shape[0]:
        raise ValueError(
            "feature_keys and values must contain the same number "
            "of memory items."
        )

    # Sum phi(k_j) v_j^T over all stored memories.
    kv_state: torch.Tensor = feature_keys.transpose(-2, -1) @ values

    # Sum phi(k_j) for the normalization denominator.
    key_state: torch.Tensor = feature_keys.sum(dim=0)

    return kv_state, key_state

In [20]:
def read_linear_memory(
    query_feature: torch.Tensor,
    kv_state: torch.Tensor,
    key_state: torch.Tensor,
    epsilon: float = 1e-6,
) -> torch.Tensor:
    """Read one query from a normalized linear-attention memory.

    Args:
        query_feature: Query feature with shape `(M)`.
        kv_state: Memory matrix with shape `(M, D_V)`.
        key_state: Normalization state with shape `(M)`.
        epsilon: Small numerical-stability constant.

    Returns:
        Retrieved value with shape `(D_V)`.
    """
    numerator: torch.Tensor = query_feature @ kv_state

    denominator: torch.Tensor = query_feature @ key_state

    return numerator / (denominator + epsilon)

In [21]:
orthogonal_keys: torch.Tensor = torch.tensor(
    [
        [1.0, 0.0],
        [0.0, 1.0],
    ]
)

values: torch.Tensor = torch.tensor(
    [
        [1.0, 0.0],
        [0.0, 1.0],
    ]
)

kv_state, key_state = build_linear_memory(
    orthogonal_keys,
    values,
)

query: torch.Tensor = orthogonal_keys[0]

retrieved: torch.Tensor = read_linear_memory(
    query,
    kv_state,
    key_state,
)

print("KV state:")
print(kv_state)

print("retrieved:")
print(retrieved)

KV state:
tensor([[1., 0.],
        [0., 1.]])
retrieved:
tensor([1.0000, 0.0000])


In [22]:
similar_keys: torch.Tensor = torch.tensor(
    [
        [1.0, 0.0],
        [0.8, 0.6],
    ]
)

kv_state, key_state = build_linear_memory(
    similar_keys,
    values,
)

query = similar_keys[0]

retrieved = read_linear_memory(
    query,
    kv_state,
    key_state,
)

print("retrieved:")
print(retrieved)

retrieved:
tensor([0.5556, 0.4444])


In [23]:
similarities = [
    0.0,
    0.25,
    0.50,
    0.75,
    0.95,
]

for overlap in similarities:
    feature_keys: torch.Tensor = torch.tensor(
        [
            [1.0, 0.0],
            [overlap, 1.0 - overlap],
        ]
    )

    kv_state, key_state = build_linear_memory(
        feature_keys,
        values,
    )

    retrieved = read_linear_memory(
        feature_keys[0],
        kv_state,
        key_state,
    )

    print(f"overlap={overlap:.2f} -> retrieved={retrieved}")

overlap=0.00 -> retrieved=tensor([1.0000, 0.0000])
overlap=0.25 -> retrieved=tensor([0.8000, 0.2000])
overlap=0.50 -> retrieved=tensor([0.6667, 0.3333])
overlap=0.75 -> retrieved=tensor([0.5714, 0.4286])
overlap=0.95 -> retrieved=tensor([0.5128, 0.4872])


### What Causes Interference?

The fixed-size state stores memories by superposition:

$$
S
=
\sum_j
\phi(k_j)v_j^\top.
$$

When querying memory item $a$,

$$
\phi(k_a)^\top S
=
\left[
\phi(k_a)^\top\phi(k_a)
\right]v_a
+
\sum_{j\neq a}
\left[
\phi(k_a)^\top\phi(k_j)
\right]v_j.
$$

The first term is the desired memory.

The remaining terms are interference.

If different keys occupy nearly orthogonal directions in feature space,
the cross terms are small.

If keys overlap strongly, their values become mixed during retrieval.

Therefore, a fixed-size associative state introduces a fundamental
memory-management problem:

> How should new information be written when the memory already contains
> information associated with a similar key?

## 7. From Additive Memory to the Delta Rule

The linear-attention state uses an additive write rule:

$$
S_t
=
S_{t-1}
+
k_t v_t^\top.
$$

This rule writes the full value into memory every time a key/value pair
appears.

To isolate the consequences of this update, consider a simplified
associative memory:

$$
W
\in
\mathbb{R}^{M\times D_V}.
$$

A key $k$ reads from the memory through

$$
\hat v
=
k^\top W.
$$

The question is:

> What should be written if the memory already predicts part, or all,
> of the desired value?

In [24]:
def read_fast_weight_memory(
    key: torch.Tensor,
    memory: torch.Tensor,
) -> torch.Tensor:
    """Read a value from a simple associative memory.

    Args:
        key: Query/key vector with shape `(M)`.
        memory: Fast-weight matrix with shape `(M, D_V)`.

    Returns:
        Predicted value with shape `(D_V)`.
    """
    return key @ memory

In [25]:
def additive_memory_update(
    memory: torch.Tensor,
    key: torch.Tensor,
    value: torch.Tensor,
) -> torch.Tensor:
    """Write a full key-value association into memory.

    Args:
        memory: Current memory with shape `(M, D_V)`.
        key: Key vector with shape `(M)`.
        value: Value vector with shape `(D_V)`.

    Returns:
        Updated memory with shape `(M, D_V)`.
    """
    update: torch.Tensor = key.unsqueeze(-1) @ value.unsqueeze(0)

    return memory + update

In [26]:
key: torch.Tensor = torch.tensor([1.0, 0.0])

value: torch.Tensor = torch.tensor([1.0, -0.5])

memory: torch.Tensor = torch.zeros(
    2,
    2,
)

for step in range(5):
    memory = additive_memory_update(
        memory,
        key,
        value,
    )

    prediction: torch.Tensor = read_fast_weight_memory(
        key,
        memory,
    )

    print(
        f"step {step + 1}:",
        prediction,
    )

step 1: tensor([ 1.0000, -0.5000])
step 2: tensor([ 2., -1.])
step 3: tensor([ 3.0000, -1.5000])
step 4: tensor([ 4., -2.])
step 5: tensor([ 5.0000, -2.5000])


In [27]:
def delta_memory_update(
    memory: torch.Tensor,
    key: torch.Tensor,
    value: torch.Tensor,
    write_rate: float = 1.0,
) -> torch.Tensor:
    """Update associative memory using the Delta Rule.

    The memory writes only the part of the target value that is not
    already predicted for the current key.

    Args:
        memory: Current memory with shape `(M, D_V)`.
        key: Key vector with shape `(M)`.
        value: Desired value with shape `(D_V)`.
        write_rate: Strength of the memory update.

    Returns:
        Updated memory with shape `(M, D_V)`.
    """
    prediction: torch.Tensor = key @ memory

    error: torch.Tensor = value - prediction

    update: torch.Tensor = key.unsqueeze(-1) @ error.unsqueeze(0)

    return memory + write_rate * update

In [28]:
memory = torch.zeros(
    2,
    2,
)

for step in range(5):
    memory = delta_memory_update(
        memory,
        key,
        value,
        write_rate=1.0,
    )

    prediction = read_fast_weight_memory(
        key,
        memory,
    )

    print(
        f"step {step + 1}:",
        prediction,
    )

step 1: tensor([ 1.0000, -0.5000])
step 2: tensor([ 1.0000, -0.5000])
step 3: tensor([ 1.0000, -0.5000])
step 4: tensor([ 1.0000, -0.5000])
step 5: tensor([ 1.0000, -0.5000])


### Additive Write vs Delta Write

The additive rule always writes the complete value:

$$
W_t
=
W_{t-1}
+
k_t v_t^\top.
$$

The Delta Rule first asks what the current memory already predicts:

$$
\hat v_t
=
k_t^\top W_{t-1},
$$

computes the prediction error

$$
e_t
=
v_t-\hat v_t,
$$

and writes only the error:

$$
W_t
=
W_{t-1}
+
\eta k_t e_t^\top.
$$

Therefore:

<pre>
additive update
→ write the target again

delta update
→ read first
→ compute prediction error
→ write only what is missing
</pre>

In [29]:
key_1: torch.Tensor = torch.tensor([1.0, 0.0])

key_2: torch.Tensor = torch.tensor([0.8, 0.6])

value_1: torch.Tensor = torch.tensor([1.0, 0.0])

value_2: torch.Tensor = torch.tensor([0.0, 1.0])

memory = torch.zeros(
    2,
    2,
)

memory = delta_memory_update(
    memory,
    key_1,
    value_1,
)

print("after writing memory 1:")

print(
    "read key 1:",
    read_fast_weight_memory(
        key_1,
        memory,
    ),
)

memory = delta_memory_update(
    memory,
    key_2,
    value_2,
)

print("after writing memory 2:")

print(
    "read key 1:",
    read_fast_weight_memory(
        key_1,
        memory,
    ),
)

print(
    "read key 2:",
    read_fast_weight_memory(
        key_2,
        memory,
    ),
)

after writing memory 1:
read key 1: tensor([1., 0.])
after writing memory 2:
read key 1: tensor([0.3600, 0.8000])
read key 2: tensor([-6.5088e-08,  1.0000e+00])


### Takeaway: The Write Rule Matters

A fixed-size memory is not defined only by its state size.

Its behavior also depends critically on how new information is written.

A pure additive update uses

$$
W_t
=
W_{t-1}
+
k_t v_t^\top.
$$

The Delta Rule instead uses the current memory prediction

$$
\hat v_t
=
W_{t-1}^\top k_t,
$$

and writes only the prediction error:

$$
W_t
=
W_{t-1}
+
\eta
k_t
(v_t-\hat v_t)^\top.
$$

This update can be interpreted as online gradient descent on

$$
\frac12
\|
v_t-W^\top k_t
\|^2.
$$

The main lesson is broader than the Delta Rule itself:

> A recurrent memory architecture is characterized not only by what it
> stores, but also by its read rule and its write rule.

## 8. Softmax Attention, FlashAttention, and Linear Attention

It is important to separate two different questions:

1. **What mathematical function does the model compute?**
2. **How is that function executed efficiently on hardware?**

These are not the same problem.

Standard softmax attention and FlashAttention compute the same attention
function.

Kernelized linear attention changes the attention computation itself.

### Standard Softmax Attention

For one attention head,

$$
Y
=
\operatorname{softmax}
\left(
\frac{QK^\top}{\sqrt D}
\right)V.
$$

The mathematical definition contains pairwise interactions between query
and key positions.

For sequence length $T$, there are on the order of

$$
T^2
$$

query-key interactions.

A naive implementation may explicitly materialize the full score or
attention matrix with shape

$$
(T,T).
$$

### FlashAttention

FlashAttention does not replace softmax attention with a different
attention mechanism.

It reorganizes the execution of exact softmax attention.

Instead of treating the full $T \times T$ attention matrix as a large
intermediate tensor that must be repeatedly written to and read from
high-bandwidth memory, the computation is tiled into smaller blocks.

Conceptually:

<pre>
standard mathematical function
QK^T
  ↓
softmax
  ↓
multiply by V

FlashAttention
same mathematical function
  ↓
different blocking and memory schedule
</pre>

Therefore, FlashAttention primarily changes the execution strategy,
not the model definition.

### FlashAttention Does Not Make Softmax Attention Linear in Sequence Length

FlashAttention reduces memory traffic and avoids materializing the full
attention matrix in the naive way.

However, it still computes exact softmax attention.

The number of query-key interactions therefore remains quadratic in
sequence length:

$$
O(T^2 D).
$$

This distinction is important:

> Better execution does not necessarily mean a different asymptotic
> mathematical workload.

### Kernelized Linear Attention

Kernelized linear attention changes the mathematical structure of
attention.

It assumes a factorized similarity:

$$
\operatorname{sim}(q,k)
=
\phi(q)^\top\phi(k).
$$

This allows key/value information to be summarized into

$$
S
=
\sum_j
\phi(k_j)v_j^\top,
$$

and

$$
z
=
\sum_j
\phi(k_j).
$$

A query can then read from the summary:

$$
y_i
=
\frac{
\phi(q_i)^\top S
}{
\phi(q_i)^\top z
}.
$$

For fixed feature and value dimensions, the sequence-length dependence
can therefore become linear rather than quadratic.

### Three Different Ideas

| Property | Standard Softmax Attention | FlashAttention | Kernelized Linear Attention |
|---|---|---|---|
| Attention function | Softmax attention | Softmax attention | Kernel / feature-map attention |
| Exact same math as standard softmax? | Yes | Yes | Generally no |
| Pairwise query-key interactions | Yes | Yes | Can be factorized away |
| Arithmetic scaling in $T$ | $O(T^2D)$ | $O(T^2D)$ | $O(TMD_V)$ for fixed feature size $M$ |
| Naive $T \times T$ intermediate | Yes | Avoided / tiled | Not required |
| Main idea | Pairwise normalized similarity | Better execution of the same attention | Different factorized attention structure |
| Recurrent fixed-size state | No | No | Yes, in the causal recurrent form |

### Attention Matrix Memory and KV-Cache Memory Are Different Problems

Avoiding a large temporary attention matrix does not automatically remove
the autoregressive KV cache.

Exact softmax autoregressive attention still needs access to historical
keys and values:

$$
K_{1:t},V_{1:t}.
$$

Its persistent inference state therefore grows with sequence length.

The recurrent form of linear attention instead keeps only a compressed
state such as

$$
S_t
$$

and

$$
z_t,
$$

whose size can remain independent of sequence length.

Therefore:

<pre>
FlashAttention
→ reduces temporary attention execution cost

GQA / MQA
→ reduce persistent KV-cache width

recurrent linear attention
→ changes historical memory into a fixed-size state
</pre>

### Three Optimization Axes

Modern attention systems can optimize different parts of the problem.

#### 1. Reduce KV width

MQA and GQA reduce

$$
H_{KV},
$$

while keeping token-wise historical keys and values.

#### 2. Improve execution

FlashAttention computes the same softmax attention more efficiently by
changing the memory-access pattern.

#### 3. Change the state representation

Recurrent linear attention summarizes history into a fixed-size state
instead of keeping every historical key and value separately.

These approaches solve different bottlenecks and should not be treated as
interchangeable.

### Takeaway: Model and Execution Are Different Layers

A model can often be understood at several separate levels:

<pre>
mathematical objective
        ↓
model architecture
        ↓
state representation
        ↓
execution algorithm
        ↓
hardware behavior
</pre>

An optimization may change only one of these levels.

FlashAttention changes execution while preserving softmax attention.

GQA changes attention parameterization and KV-cache structure while
preserving the basic softmax-attention computation.

Kernelized linear attention changes the attention formulation itself and
admits a different recurrent state representation.

Recognizing which level changed is often more useful than memorizing the
name of a new method.

## 9. Final Takeaways

This lesson began with a simple question:

> Can attention avoid explicitly performing all pairwise interactions
> between sequence positions?

Standard softmax attention computes

$$
Y
=
\operatorname{softmax}
\left(
\frac{QK^\top}{\sqrt D}
\right)V.
$$

For sequence length $T$, the query-key interaction contains on the order
of

$$
T^2
$$

pairwise comparisons.

The key observation behind kernelized linear attention is that if the
similarity function can be factorized as

$$
\operatorname{sim}(q,k)
=
\phi(q)^\top \phi(k),
$$

then attention can be reorganized using associativity.

Instead of explicitly computing all pairwise similarities,

$$
\Phi(Q)\Phi(K)^\top,
$$

we can first summarize the keys and values:

$$
S
=
\Phi(K)^\top V,
$$

and

$$
z
=
\sum_j \phi(k_j).
$$

A query then reads from this summary:

$$
y_i
=
\frac{
\phi(q_i)^\top S
}{
\phi(q_i)^\top z
}.
$$

This removes the explicit $T\times T$ interaction matrix.

### Causal linear attention becomes a recurrent state

For autoregressive modeling, only the prefix

$$
j\le t
$$

may be visible.

The memory therefore becomes

$$
S_t
=
\sum_{j\le t}
\phi(k_j)v_j^\top,
$$

and

$$
z_t
=
\sum_{j\le t}
\phi(k_j).
$$

These states can be updated recurrently:

$$
S_t
=
S_{t-1}
+
\phi(k_t)v_t^\top,
$$

$$
z_t
=
z_{t-1}
+
\phi(k_t).
$$

The current query reads

$$
y_t
=
\frac{
\phi(q_t)^\top S_t
}{
\phi(q_t)^\top z_t
}.
$$

This gives two equivalent execution views:

<pre>
training
full sequence
    ↓
prefix states

inference
one token at a time
    ↓
recurrent state update
</pre>

We verified that three implementations of the same kernel attention
mechanism agree numerically:

<pre>
explicit pairwise causal attention
            =
recurrent causal attention
            =
parallel prefix causal attention
</pre>

They are different execution strategies for the same mathematical
mechanism.

### Fixed-size state changes the memory representation

Standard autoregressive attention keeps historical keys and values
individually addressable:

$$
K_{1:t},V_{1:t}.
$$

Its KV cache therefore grows with sequence length.

The recurrent linear-attention formulation instead compresses history
into

$$
S_t
$$

and

$$
z_t.
$$

For fixed feature and value dimensions, the persistent inference state
does not grow with $T$.

This is not a lossless compression of the KV cache.

Historical information is superposed into a shared state.

### Superposition creates interference

The memory state is

$$
S
=
\sum_j
\phi(k_j)v_j^\top.
$$

Reading with query $\phi(q)$ gives

$$
\phi(q)^\top S
=
\sum_j
\left[
\phi(q)^\top\phi(k_j)
\right]v_j.
$$

If different keys overlap strongly in feature space, multiple values
contribute to the same read.

Therefore, fixed-size memory introduces a new problem:

$$
\text{memory interference}.
$$

Reducing memory growth is not free.

### The write rule matters

Pure linear attention uses an additive memory update:

$$
W_t
=
W_{t-1}
+
k_t v_t^\top.
$$

This repeatedly writes the entire target value even when the memory
already predicts it correctly.

The Delta Rule instead computes

$$
\hat v_t
=
W_{t-1}^\top k_t,
$$

then the prediction error

$$
e_t
=
v_t-\hat v_t,
$$

and updates

$$
W_t
=
W_{t-1}
+
\eta k_t e_t^\top.
$$

This is equivalent to online gradient descent on

$$
\frac12
\|
v_t-W^\top k_t
\|^2.
$$

This gives a broader way to analyze recurrent memory systems:

<pre>
state representation
        +
read rule
        +
write rule
        +
normalization
</pre>

### Linear attention is not FlashAttention

It is important to separate model mathematics from execution.

Standard softmax attention computes

$$
\operatorname{softmax}(QK^\top)V.
$$

FlashAttention computes the same mathematical attention function using a
more efficient tiled execution strategy.

Kernelized linear attention changes the attention formulation itself.

Therefore:

<pre>
FlashAttention
→ same attention mathematics
→ different execution strategy

GQA / MQA
→ same basic softmax attention
→ fewer KV heads
→ smaller KV cache

kernelized linear attention
→ different similarity structure
→ different historical state representation
</pre>

### Complexity summary

| Mechanism | Sequence computation | Persistent autoregressive state | Historical representation |
|---|---|---|---|
| Standard softmax attention | $O(T^2D)$ | grows with $T$ | individual K/V |
| GQA / MQA | still quadratic attention | grows with $T$, but narrower | individual K/V |
| FlashAttention | still quadratic arithmetic | grows with $T$ | individual K/V |
| Recurrent linear attention | $O(TMD_V)$ for fixed feature size | independent of $T$ | compressed state |

Here $M$ is the feature-map dimension.

The exact practical cost also depends on implementation, hardware,
memory traffic, batching, and kernel efficiency.

### The central lesson

The most important idea is not that linear attention is universally
better than softmax attention.

The important idea is that attention can be understood as a memory
system.

Changing

$$
\text{how history is represented}
$$

changes both computational cost and model behavior.

A useful way to analyze a new sequence architecture is therefore to ask:

<pre>
What is the state?

How is information written?

How is information read?

What is normalized?

What information remains individually addressable?

How does state size scale with sequence length?

What mathematical function changed?

What changed only in the implementation?
</pre>

### Mental Model

Standard attention:

<pre>
store the past
        ↓
search the past
</pre>

Recurrent linear attention:

<pre>
compress the past into a state
        ↓
update the state
        ↓
query the state
</pre>

The first keeps history explicitly addressable.

The second trades explicit history for a fixed-size dynamic memory.